# Modul 13: Klassische Modelle für Bild- und Signaldaten | Übungen

## Überblick

Sie wandeln Bilder in Pixel-, Histogramm- und kantenorientierte Merkmale um und bewerten klassische Bildklassifikatoren. Anschließend zerlegen Sie Signale in gelabelte Fenster, extrahieren statistische und frequenzbasierte Merkmale und führen zeitlich korrekte Modellvergleiche durch.

**Zugehörige Vorlesungen**

- **Klassische Bildmodelle**
- **Signalmodelle**

## Lernziele

Nach der Bearbeitung können Sie:

- kleine Graustufenbilder als Pixel-, Histogramm- und HOG-ähnliche Merkmale vorbereiten.
- klassische Bildmodelle leakage-frei trainieren und Fehlklassifikationen sichtbar untersuchen.
- Signalfenster mit Zeit- und Frequenzmerkmalen beschreiben und zeitlich korrekt bewerten.

## Geprüfte Fähigkeiten

- Digits-Bildtensoren, Pixelmerkmale, regionale Histogramme und einfache HOG-Merkmale
- Pipelines, PCA, Konfusionsmatrix und visualisierte Modellfehler
- Fensterung, statistische Signalmerkmale, FFT, zeitlicher Split und Baselines

## Hinweise zur Bearbeitung

Dieses Notebook dient als praktische Übung und Lernstandskontrolle. Führen Sie zuerst die Einrichtungszelle aus und bearbeiten Sie danach die Aufgaben in der angegebenen Reihenfolge. Die vorgesehenen Arbeitsbereiche sind deutlich markiert.

- **Erwarteter Schwierigkeitsgrad:** fortgeschritten
- Verwenden Sie sprechende Variablennamen und prüfen Sie wichtige Zwischenformen und Wertebereiche.
- Verändern Sie die vorgegebenen Zufalls-Startwerte nur, wenn eine Aufgabe dies ausdrücklich verlangt.
- Interpretieren Sie Ergebnisse fachlich. Eine einzelne Kennzahl ist selten eine vollständige Begründung.
- Alle Aufgaben sind für die kostenlose Google-Colab-Umgebung ausgelegt. Die Datensätze und Modelle sind bewusst klein gehalten. Eine GPU ist nicht erforderlich, kann aber bei einzelnen Deep-Learning-Aufgaben die Laufzeit verkürzen.

## Einrichtung und gemeinsame Datenbasis

Die Setup-Zelle lädt den kleinen Digits-Datensatz und erzeugt ein synthetisches Sensorsignal mit zeitlich wechselnden Zuständen. Alle Berechnungen laufen auf der CPU und benötigen keine externen Dateien.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

ziffern = load_digits()
bilder = ziffern.images.astype(np.float64)
bild_labels = ziffern.target

bild_train, bild_test, label_train, label_test = train_test_split(
    bilder,
    bild_labels,
    test_size=0.25,
    stratify=bild_labels,
    random_state=RANDOM_SEED,
)

# Synthetisches Signal: drei Zustände mit unterschiedlichen dominanten Frequenzen.
abtastrate_hz = 50
segment_laenge = 250
zustandsfolge = [0, 1, 2, 0, 2, 1, 0, 1, 2, 0, 2, 1]
signal_abschnitte = []
signal_labels = []
for segment_index, zustand in enumerate(zustandsfolge):
    lokale_zeit = np.arange(segment_laenge) / abtastrate_hz
    frequenz = [1.5, 4.0, 8.0][zustand]
    amplitude = [1.0, 0.8, 0.55][zustand]
    abschnitt = amplitude * np.sin(2 * np.pi * frequenz * lokale_zeit)
    abschnitt += 0.15 * rng.normal(size=segment_laenge)
    abschnitt += 0.05 * segment_index
    signal_abschnitte.append(abschnitt)
    signal_labels.extend([zustand] * segment_laenge)

sensor_signal = np.concatenate(signal_abschnitte)
sensor_label = np.asarray(signal_labels)
zeit_s = np.arange(sensor_signal.size) / abtastrate_hz

print("Bilder:", bilder.shape)
print("Signalpunkte:", sensor_signal.shape)

### Aufgabe 1: Bilddaten prüfen und Pixelmerkmale vorbereiten

Visualisieren Sie je ein Trainingsbild der Klassen 0 bis 4. Wandeln Sie anschließend die 8-mal-8-Bilder in 64 Pixelmerkmale um und skalieren Sie die Pixelwerte in den Bereich 0 bis 1.

Trainieren Sie eine Pipeline aus StandardScaler und logistischer Regression. Berichten Sie Accuracy und Macro-F1 auf dem Testdatensatz.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Welche Voraussetzungen müssen für rohe Pixelmerkmale ungefähr erfüllt sein?

### Aufgabe 2: Regionale Histogramme als kompakte Bildmerkmale

Schreiben Sie eine Funktion `regionale_histogramme`, die jedes 8-mal-8-Bild in vier 4-mal-4-Quadranten teilt. Berechnen Sie je Quadrant ein Histogramm mit vier festen Bins im Wertebereich 0 bis 16 und normieren Sie es auf Summe 1.

Erzeugen Sie damit 16 Merkmale pro Bild, trainieren Sie dieselbe Modellart wie zuvor und vergleichen Sie Leistung und Merkmalsanzahl mit der Pixelbaseline.

In [ ]:
def regionale_histogramme(bild_stapel):
    # Erwartete Eingabeform: (Anzahl, 8, 8)
    # Erwartete Ausgabeform: (Anzahl, 16)
    pass

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Was geht bei globalen oder regionalen Histogrammen gegenüber Pixelmerkmalen verloren?

### Aufgabe 3: Einfache HOG-Merkmale und PCA vergleichen

Implementieren Sie eine vereinfachte HOG-Funktion. Berechnen Sie horizontale und vertikale Gradienten mit `np.gradient`, daraus Betrag und Orientierung im Bereich 0 bis 180 Grad. Teilen Sie das Bild in vier 4-mal-4-Zellen und erstellen Sie je Zelle ein gewichtetes Orientierungshistogramm mit neun Bins.

Trainieren Sie ein Modell auf den 36 HOG-Merkmalen. Vergleichen Sie es außerdem mit einer Pipeline aus skalierten Pixelmerkmalen, PCA mit 20 Komponenten und logistischer Regression.

In [ ]:
from sklearn.decomposition import PCA

def einfache_hog_merkmale(bild_stapel, anzahl_bins=9):
    pass

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Warum muss PCA innerhalb der Pipeline auf den Trainingsdaten angepasst werden?

### Aufgabe 4: Fehlklassifikationen mit Konfusionsmatrix und Bildern analysieren

Verwenden Sie das leistungsstärkste der drei Bildmodelle nach Test-Accuracy. Erstellen Sie eine Konfusionsmatrix. Bestimmen Sie anschließend das häufigste Verwechslungspaar außerhalb der Hauptdiagonalen und visualisieren Sie bis zu sechs entsprechende falsch klassifizierte Testbilder.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Warum ist das Lesen konkreter Fehlbilder wertvoller als nur die Gesamt-Accuracy?

### Aufgabe 5: Signale in Fenster zerlegen und Merkmale extrahieren

Zerlegen Sie `sensor_signal` ohne Überlappung in Fenster von 100 Messpunkten. Weisen Sie jedem Fenster das Mehrheitslabel seiner Messpunkte zu.

Berechnen Sie je Fenster mindestens diese Merkmale: Mittelwert, Standardabweichung, Minimum, Maximum, RMS, Peak-to-Peak, dominante positive Frequenz und spektralen Schwerpunkt. Speichern Sie alles in einem DataFrame und prüfen Sie Form, Fehlwerte und Klassenverteilung.

In [ ]:
def extrahiere_signalmerkmale(signal_fenster, sampling_rate):
    pass

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Welche Wirkung hätte eine größere Fensterlänge auf Zeitauflösung und Frequenzschätzung?

### Aufgabe 6: Integrationsaufgabe: zeitlich korrektes Signalmodell

Sortieren Sie die Fenstertabelle nach `Startindex`. Verwenden Sie die ersten 70 Prozent der Fenster als Training und die letzten 30 Prozent als Test, ohne zufälliges Mischen.

Vergleichen Sie einen `DummyClassifier(strategy="most_frequent")` mit einer Pipeline aus StandardScaler und logistischer Regression. Berechnen Sie Accuracy und Macro-F1, zeichnen Sie wahre und vorhergesagte Zustände über der Testzeit und erläutern Sie, warum ein zufälliger Split hier problematisch wäre.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Warum kann ein zufälliger Fenstersplit zu unrealistisch guten Ergebnissen führen?

## Abschlusskontrolle

Prüfen Sie vor dem Abschluss:

- Lassen sich alle Zellen in sinnvoller Reihenfolge ausführen?
- Sind Formen, Datentypen, Wertebereiche und Zufalls-Startwerte dokumentiert?
- Wurden Trainings-, Validierungs- und Testinformationen sauber getrennt?
- Sind Diagramme und Kennzahlen beschriftet und fachlich interpretiert?
- Können Sie erklären, warum die gewählten Methoden zur Aufgabenstellung passen?